# Day 5 — Advanced SQL, Performance, Optimization and Interview Project

**Target:** 3–4 years experience SQL hands-on practice.

**Database:** MySQL 8.x. The tables/data are created in a separate setup SQL file; this notebook focuses on syllabus, commands, explanations, practice queries, assignments, and solutions.


## Learning method

1. Run each query manually in MySQL Workbench/CLI.
2. Before executing, predict the result.
3. Change one condition and rerun.
4. Check execution plans where applicable.
5. Complete assignments without looking at the solution first.


## Important execution commands

```sql
SHOW DATABASES;
USE company_sql_practice;
SHOW TABLES;
SELECT DATABASE();
DESCRIBE employees;
```

The setup file contains the five-table schema and large sample dataset.


## Syllabus

- Indexes: single-column and composite
- Primary, unique, foreign-key indexes
- EXPLAIN and EXPLAIN ANALYZE
- Query execution plans
- SARGable predicates
- Index-friendly filtering
- CTE vs derived table vs subquery
- Temporary tables
- Views
- Recursive CTEs
- Hierarchical employee reporting
- Transactions and isolation concepts
- Deadlock awareness
- SQL security and parameterized queries
- Data quality and reconciliation checks
- Pagination strategies
- Advanced interview patterns
- Capstone: employee + customer + order + project analytics


## Practice Queries

Each query is intentionally written as a workplace-style example. Read the explanation before changing it.


### Practice 1

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Index inspection
SHOW INDEX FROM employees;
SHOW INDEX FROM orders;
SHOW INDEX FROM order_items;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 2

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Explain a query
EXPLAIN
SELECT e.employee_id, e.employee_name, e.salary
FROM employees e
WHERE e.department_id = 3
  AND e.salary >= 100000
ORDER BY e.salary DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 3

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Composite index practice (create only after checking existing indexes)
CREATE INDEX idx_emp_dept_salary
ON employees(department_id, salary);


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 4

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- View
CREATE OR REPLACE VIEW v_customer_revenue AS
SELECT c.customer_id, c.customer_name,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.customer_name;

SELECT *
FROM v_customer_revenue
ORDER BY revenue DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 5

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Recursive CTE: employee hierarchy
WITH RECURSIVE org AS (
    SELECT employee_id, employee_name, manager_id, 0 AS level_no
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    SELECT e.employee_id, e.employee_name, e.manager_id, o.level_no + 1
    FROM employees e
    JOIN org o ON e.manager_id = o.employee_id
)
SELECT *
FROM org
ORDER BY level_no, employee_id;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 6

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Keyset pagination pattern
SELECT employee_id, employee_name, salary
FROM employees
WHERE (salary, employee_id) < (120000, 5000)
ORDER BY salary DESC, employee_id DESC
LIMIT 20;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 7

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Reconciliation check
SELECT
    (SELECT COUNT(*) FROM orders) AS order_count,
    (SELECT COUNT(DISTINCT order_id) FROM order_items) AS orders_with_items,
    (SELECT SUM(quantity * unit_price) FROM order_items) AS item_revenue;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


## Assignments

1. Use EXPLAIN to compare an indexed and non-indexed predicate.
2. Design a composite index for a common department + salary query and explain column order.
3. Build a recursive organization chart.
4. Create a monthly revenue report with window functions.
5. Find customers whose lifetime revenue exceeds the average customer lifetime revenue.
6. Build a final 10-question interview test combining joins, CTEs, windows, procedures and optimization.
7. Capstone: produce an employee-performance and customer-revenue dashboard query set.


## Solutions

Try the assignment first. Then compare with the reference pattern below. Adapt names/filters to your dataset.


### Solution 1


In [ ]:
EXPLAIN SELECT employee_id,employee_name,salary
FROM employees
WHERE department_id=3 AND salary>=100000
ORDER BY salary DESC;


### Solution 2


In [ ]:
CREATE INDEX idx_emp_dept_salary
ON employees(department_id,salary);
-- Re-run EXPLAIN and compare access type, possible_keys, key, rows and Extra.


### Solution 3


In [ ]:
WITH RECURSIVE org AS (
 SELECT employee_id,employee_name,manager_id,0 level_no
 FROM employees WHERE manager_id IS NULL
 UNION ALL
 SELECT e.employee_id,e.employee_name,e.manager_id,o.level_no+1
 FROM employees e JOIN org o ON e.manager_id=o.employee_id
)
SELECT * FROM org ORDER BY level_no,employee_id;


### Solution 4


In [ ]:
WITH m AS (
 SELECT DATE_FORMAT(o.order_date,'%Y-%m') month,
        SUM(oi.quantity*oi.unit_price) revenue
 FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
 GROUP BY DATE_FORMAT(o.order_date,'%Y-%m')
)
SELECT month,revenue,
       LAG(revenue) OVER(ORDER BY month) prev_revenue,
       revenue-LAG(revenue) OVER(ORDER BY month) change_amount
FROM m;


### Solution 5


In [ ]:
WITH customer_rev AS (
 SELECT c.customer_id,
        SUM(oi.quantity*oi.unit_price) lifetime_revenue
 FROM customers c JOIN orders o ON o.customer_id=c.customer_id
 JOIN order_items oi ON oi.order_id=o.order_id
 GROUP BY c.customer_id
)
SELECT * FROM customer_rev
WHERE lifetime_revenue > (SELECT AVG(lifetime_revenue) FROM customer_rev);


### Solution 6


In [ ]:
-- Interview test: solve without looking at earlier examples.
-- 1) top-N per group
-- 2) anti-join
-- 3) recursive hierarchy
-- 4) month-over-month LAG
-- 5) CTE + HAVING
-- 6) procedure with OUT
-- 7) cursor + handler
-- 8) transaction rollback
-- 9) EXPLAIN/index
-- 10) duplicate diagnosis


### Solution 7


In [ ]:
-- Capstone pattern: build separate query modules for:
-- employee performance, department payroll, project allocation,
-- customer lifetime value, monthly revenue, top products, retention,
-- data-quality reconciliation, and an indexed reporting query.


## Interview Questions & Answers

**Q: WHERE vs HAVING?**  
A: WHERE filters rows before grouping; HAVING filters grouped results.

**Q: RANK vs DENSE_RANK?**  
A: Both assign equal ranks to ties; RANK leaves gaps after ties, DENSE_RANK does not.

**Q: Why can joins create duplicates?**  
A: A one-to-many relationship legitimately creates multiple result rows per parent. First define the intended row grain, then aggregate/deduplicate as required.

**Q: Cursor or set-based SQL?**  
A: Prefer set-based operations for bulk relational transformations; use cursors when row-by-row procedural behavior is genuinely required.

**Q: What does EXPLAIN provide?**  
A: It exposes the optimizer's planned access path, including table access, candidate indexes, selected index, estimated rows and other plan details.


## 5-Day Deliverable Checklist

- [ ] All practice queries executed manually
- [ ] All assignments solved without copying
- [ ] Solutions compared and rewritten
- [ ] At least 10 query variations created
- [ ] At least 5 window-function problems solved
- [ ] At least 3 stored procedures created
- [ ] At least 1 cursor + handler implemented
- [ ] Transactions tested with COMMIT and ROLLBACK
- [ ] EXPLAIN used before/after an index
- [ ] Final capstone query set completed
